In [32]:
import torch
import torch.nn as nn

#nn.Linear(input_dim,output_dim)=一个线性层
layer=nn.Linear(in_features=2,out_features=1)

x=torch.tensor([[1.0,2.0]])
y=layer(x)

print("输入shape:",x.shape)
print("输出shape:",y.shape)
print("权重shape:",layer.weight.shape)
print("偏置shape:",layer.bias.shape)

# 它内部就是：y = x @ w.T + b
# 和你之前手写的 wx + b 一模一样，只是封装好了

输入shape: torch.Size([1, 2])
输出shape: torch.Size([1, 1])
权重shape: torch.Size([1, 2])
偏置shape: torch.Size([1])


In [41]:
class LogisticRegression(nn.Module):
    def __init__(self,input_dim):
        super().__init__()
        #就一层：线性变换
        self.linear=nn.Linear(input_dim,1)
        #Sigmoid也封装好了
        self.sigmoid=nn.Sigmoid()

    def forward(self,x):
        #前向传播：定义数据怎么流
        z=self.linear(x) #z=wx+b
        y_prob=self.sigmoid(z)#压到0～1
        return y_prob

#创建模型
model=LogisticRegression(input_dim=2)
print(model)

#测试前向
x_test=torch.tensor([[0.5,-1.2]])
output=model(x_test)
print("预测概率:",output.item())


LogisticRegression(
  (linear): Linear(in_features=2, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
预测概率: 0.7343581914901733


In [42]:
#损失函数：交叉熵
criterion=nn.BCELoss() #Binary Cross Entropy

#优化器：自动帮你做w=w-lr*grad
optimizer=torch.optim.SGD(model.parameters(),lr=0.1)
#注意：model.parameters()自动把所有w和b交给优化器


In [44]:
#假数据（2分类）
X=torch.tensor([[1.0,2.0],[2.0,1.0],[3.0,4.0],[4.0,3.0]])
y=torch.tensor([[0.0],[0.0],[1.0],[1.0]])

model=LogisticRegression(input_dim=2)
criterion=nn.BCELoss()
optimizer=torch.optim.SGD(model.parameters(),lr=0.1)

# ========== 训练循环（5步模板，所有深度学习都这样写）==========
epochs = 1000
for epoch in range(epochs):
    # Step 1: 前向传播
    y_pred = model(X)
    
    # Step 2: 算损失
    loss = criterion(y_pred, y)
    
    # Step 3: 清零旧梯度（必须！）
    optimizer.zero_grad()
    
    # Step 4: 反向传播（自动算梯度）
    loss.backward()
    
    # Step 5: 更新参数（自动做 w = w - lr * grad）
    optimizer.step()
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch}: loss = {loss.item():.4f}")

# 训练完看结果
print("\n最终预测:")
print(model(X).detach().numpy())

Epoch 0: loss = 0.9569
Epoch 100: loss = 0.4054
Epoch 200: loss = 0.2868
Epoch 300: loss = 0.2170
Epoch 400: loss = 0.1724
Epoch 500: loss = 0.1420
Epoch 600: loss = 0.1202
Epoch 700: loss = 0.1039
Epoch 800: loss = 0.0913
Epoch 900: loss = 0.0813

最终预测:
[[0.0962134]
 [0.0963352]
 [0.9557678]
 [0.955827 ]]


In [51]:
import torch
import torch.nn as nn
import numpy as np

#设置随机种子
torch.manual_seed(42)
np.random.seed(42)

#100个样本，2个特征，二分类
n_samples=100
n_features=2

#生成两类数据
X_class0 =torch.randn(50,2)+torch.tensor([2.0,2.0]) #中心在（2，2）
X_class1 =torch.randn(50,2)+torch.tensor([-2.0,-2.0]) #中心在（-2，-2）

X=torch.cat([X_class0,X_class1],dim=0)
y=torch.cat([torch.zeros(50,1), #类别0
             torch.ones(50,1)], #类别1
            dim=0)              #（100，1）

print("X shape:",X.shape)
print("y shape:",y.shape)
print("前5个样本:\n",torch.cat([X[:5],y[:5]],dim=1))


X shape: torch.Size([100, 2])
y shape: torch.Size([100, 1])
前5个样本:
 tensor([[ 3.9269,  3.4873,  0.0000],
        [ 2.9007, -0.1055,  0.0000],
        [ 2.6784,  0.7655,  0.0000],
        [ 1.9569,  0.3953,  0.0000],
        [ 1.2479,  3.6487,  0.0000]])


In [60]:
class LogisticRegression(nn.Module):
    def __init__(self,input_dim):
        super().__init__()
        #就一层线性变换+sigmoid
        self.linear=nn.Linear(input_dim,1)
        self.sigmoid=nn.Sigmoid()

    def forward(self,x):
        z=self.linear(x)     #z = wx + b
        return self.sigmoid(z)  #压到0~1

#创建模型
model=LogisticRegression(input_dim=2)

#看模型参数
for name,param in model.named_parameters():
    print(f"{name}:shape{param.shape}")

linear.weight:shapetorch.Size([1, 2])
linear.bias:shapetorch.Size([1])


In [54]:
#损失函数：二分类交叉熵
criterion=nn.BCELoss()

#优化器：自动做 w = w - lr*grad
#model.parameters()自动把所有w和b交给优化器
optimizer= torch.optim.SGD(model.parameters(),lr=0.1)

In [57]:
epochs=1000

for epoch in range(epochs):
    #step 1:前向传播
    y_pred=model(X)
    #step 2:算损失
    loss=criterion(y_pred,y)
    #step 3:清零旧梯度（必须！Pytorch默认累加）
    optimizer.zero_grad()
    #step 4:反向传播（自动算梯度）
    loss.backward()
    #step 5:更新参数
    optimizer.step()

    #每100轮打印一次
    if epoch%100==0:
        #算准确率
        y_pred_label=(y_pred>=0.5).float()
        accuracy=(y_pred_label==y).float().mean()
        print(f"Epoch{epoch:4d}:Loss={loss.item():.4f},Acc={accuracy.item():.4f}")

print("\n训练完成！")


Epoch   0:Loss=1.4844,Acc=0.0200
Epoch 100:Loss=0.0276,Acc=1.0000
Epoch 200:Loss=0.0163,Acc=1.0000
Epoch 300:Loss=0.0120,Acc=1.0000
Epoch 400:Loss=0.0097,Acc=1.0000
Epoch 500:Loss=0.0082,Acc=1.0000
Epoch 600:Loss=0.0071,Acc=1.0000
Epoch 700:Loss=0.0064,Acc=1.0000
Epoch 800:Loss=0.0058,Acc=1.0000
Epoch 900:Loss=0.0053,Acc=1.0000

训练完成！


In [58]:
#最终预测
with torch.no_grad(): #预测时不需要算梯度
    y_prob=model(X)
    y_pred=(y_prob>=0.5).float()
    final_acc=(y_pred==y).float().mean()

print(f"最终准确率:{final_acc.item():.2%}")

#看看学出来的权重
print(f"\n学出来的权重 w:{model.linear.weight.data.numpy()}")
print(f"学出来的偏置 b:{model.linear.bias.data.numpy()}")
    

最终准确率:100.00%

学出来的权重 w:[[-2.0111334 -1.9725711]]
学出来的偏置 b:[0.10237796]
